<a href="https://colab.research.google.com/github/AbhinavKumar0000/Machine_learning/blob/main/BiLSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional

In [2]:
# Simulating a small corpus of text for demonstration
data = """The quick brown fox jumps over the lazy dog
          I am a senior machine learning engineer
          Deep learning models require clean data
          The sky is blue and the sun is bright
          I love coding in python and tensorflow"""

# Split data into lines
corpus = data.lower().split("\n")

In [3]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)
total_words = len(tokenizer.word_index) + 1 # +1 for OOV token

In [4]:
# Create input sequences (N-grams)
input_sequences = []
for line in corpus:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

In [5]:
input_sequences

[[1, 6],
 [1, 6, 7],
 [1, 6, 7, 8],
 [1, 6, 7, 8, 9],
 [1, 6, 7, 8, 9, 10],
 [1, 6, 7, 8, 9, 10, 1],
 [1, 6, 7, 8, 9, 10, 1, 11],
 [1, 6, 7, 8, 9, 10, 1, 11, 12],
 [2, 13],
 [2, 13, 14],
 [2, 13, 14, 15],
 [2, 13, 14, 15, 16],
 [2, 13, 14, 15, 16, 3],
 [2, 13, 14, 15, 16, 3, 17],
 [18, 3],
 [18, 3, 19],
 [18, 3, 19, 20],
 [18, 3, 19, 20, 21],
 [18, 3, 19, 20, 21, 22],
 [1, 23],
 [1, 23, 4],
 [1, 23, 4, 24],
 [1, 23, 4, 24, 5],
 [1, 23, 4, 24, 5, 1],
 [1, 23, 4, 24, 5, 1, 25],
 [1, 23, 4, 24, 5, 1, 25, 4],
 [1, 23, 4, 24, 5, 1, 25, 4, 26],
 [2, 27],
 [2, 27, 28],
 [2, 27, 28, 29],
 [2, 27, 28, 29, 30],
 [2, 27, 28, 29, 30, 5],
 [2, 27, 28, 29, 30, 5, 31]]

In [6]:
max_sequence_len = max([len(x) for x in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

In [7]:
X, y = input_sequences[:,:-1], input_sequences[:,-1]

In [8]:
y = to_categorical(y, num_classes=total_words)

print(f"Vocab Size: {total_words}")
print(f"X Shape: {X.shape}")
print(f"y Shape: {y.shape}")

Vocab Size: 32
X Shape: (33, 8)
y Shape: (33, 32)


In [9]:
model = Sequential()

# Embedding Layer
# Input dim = Vocab size, Output dim = Embedding vector size, Input Length = Sequence length - 1 (label)
model.add(Embedding(total_words, 100, input_length=max_sequence_len-1))

# Bidirectional LSTM Layer
# We use Bidirectional() wrapper. Note: 150 units becomes 300 outputs because it concatenates both directions.
model.add(Bidirectional(LSTM(150)))
model.add(Dropout(0.2)) # Regularization to prevent overfitting

# Output Layer
model.add(Dense(total_words, activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [10]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [11]:
history = model.fit(X, y, epochs=100, verbose=1)

Epoch 1/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 4s 43ms/step - accuracy: 0.0306 - loss: 3.4669
Epoch 2/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.1531 - loss: 3.4541
Epoch 3/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.0919 - loss: 3.4447
Epoch 4/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.0919 - loss: 3.4395
Epoch 5/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.0919 - loss: 3.4305
Epoch 6/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.0919 - loss: 3.4270
Epoch 7/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.0919 - loss: 3.4236
Epoch 8/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.0919 - loss: 3.4208
Epoch 9/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.0919 - loss: 3.4118
Epoch 10/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.0919 - loss: 3.4049
Epoch 11/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.0919 - loss: 3.3989
Epoch 12/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.0919 - lo

In [12]:
seed_text = "The quick brown"
next_words = 3

for _ in range(next_words):
    token_list = tokenizer.texts_to_sequences([seed_text])[0]
    token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
    predicted = np.argmax(model.predict(token_list, verbose=0), axis=-1)

    output_word = ""
    for word, index in tokenizer.word_index.items():
        if index == predicted:
            output_word = word
            break
    seed_text += " " + output_word

print(f"\nGenerated Text: '{seed_text}'")


Generated Text: 'The quick brown fox fox over'
